# ch08 练习 —— 排序、排名与采样

> 数据：`device_defects.csv`（270 x 13）、`load_curve.csv`（6480 x 5）
> 已备好：`BIG`（2 万行 / 5 组，专门用来测稳定性）、`vals`（含并列的小样本）、
> `stable_check()`、`topn_by_sort()`、`probe()`

**做题规则**：从上往下依次填，每个 `____` 处跑通再往下。
验收块 `assert` 不要改，它是你自己对账用的。

| 题号 | 考点 |
|---|---|
| TODO(1) | `sort_values` 方向与 `na_position` |
| TODO(2) | 排序稳定性（`kind`） |
| TODO(3) | 多列排序 + 混合升降序 + 长度校验 |
| TODO(4) | 排序后索引：`loc` vs `iloc` |
| TODO(5) | `rank` 的五种 `method` |
| TODO(6) | 组内名次与组内百分位 |
| TODO(7) | `nlargest` / `nsmallest` / `keep` |
| TODO(8) | `sample` 的可复现性与 `frac` 取整 |
| TODO(9) | `sample(weights=...)` 的两道门槛 |
| TODO(10) | `value_counts` 的 `dropna` / `normalize` |
| 综合① | 组内冠军表 + 三方对账 |
| 综合② | 可复现采样报告 |

共 12 个挖空块。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

dev = pd.read_csv(DATA / "device_defects.csv")
load = pd.read_csv(DATA / "load_curve.csv")

KEY = "台区编号"
VAL = "负荷值"
CAT = "设备类型"

print("pandas", pd.__version__)
print("dev", dev.shape, "| load", load.shape)
print("负荷值缺失:", int(dev[VAL].isna().sum()))

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def stable_check(frame, by, kind):
    """稳定排序判据：同一取值内部，原始「序号」必须严格递增。

    不稳定的排序只保证「值有序」，并列行之间的先后是**未定义的**。
    """
    ordered = frame.sort_values(by, kind=kind)
    pos = ordered["序号"].to_numpy()
    keys = ordered[by].to_numpy()
    same = keys[1:] == keys[:-1]
    return bool(np.all(pos[1:][same] > pos[:-1][same]))


def topn_by_sort(frame, by, n):
    """分组 TopN 的推荐写法：整体排序 + head，比 groupby.apply 快且结果确定。"""
    return (
        frame.sort_values(by, ascending=False, kind="stable")
        .groupby(KEY, sort=False)
        .head(n)
    )


# 演示稳定性用的「大量并列」数据：2 万行、只有 5 个取值
_rng = np.random.default_rng(0)
BIG = pd.DataFrame({"分组": _rng.integers(0, 5, 20_000)})
BIG["序号"] = np.arange(len(BIG))

# 演示 rank 的 method 差异用的小样本（有 2 组并列）
vals = pd.Series([10, 20, 20, 30, 40, 40, 40, 50])
METHODS = ["average", "min", "max", "first", "dense"]

print("BIG", BIG.shape, "| 组数", BIG["分组"].nunique(), "| 平均每组",
      len(BIG) // BIG["分组"].nunique(), "行")

---

## 基础题

In [ ]:
# TODO(1)：排序方向与缺失位置：
#           by_val_desc = 按 负荷值 降序；by_val_asc = 升序
#           by_val_first = 降序 + na_position="first"
#           三者前 3 个负荷值分别存入 head_desc / head_asc
#           再记录 desc 末 3 行里 NaN 的个数 → tail_nan_desc
# 提示：dev.sort_values(VAL, ascending=False) ；na_position="first"
by_val_desc = ____
by_val_asc = ____
by_val_first = ____
head_desc = ____
head_asc = ____
tail_nan_desc = ____
# ---- 验收 ----
assert head_desc == [9999.0, 9999.0, 9999.0], f"降序前 3 应是哨兵 9999，实际 {head_desc}"
assert head_asc == [-1.0, -1.0, -1.0], f"升序前 3 应是哨兵 -1，实际 {head_asc}"
assert tail_nan_desc == 3, "na_position 默认 'last'，降序时 NaN 仍在末尾"
assert int(by_val_first[VAL].head(3).isna().sum()) == 3, "na_position='first' 时 NaN 排最前"
assert by_val_desc.index[:3].tolist() == [50, 46, 89], "排序不重置索引"
assert bool(by_val_desc.index.is_monotonic_increasing) is False
assert int(dev[VAL].isna().sum()) == 21
print("TODO(1) 通过 ->", head_desc, "|", head_asc, "| 索引", by_val_desc.index[:3].tolist())

In [ ]:
# TODO(2)：排序稳定性：用 stable_check 检测四种 kind 在 BIG 上的表现，
#           结果存入 quick_stable / heap_stable / merge_stable / stable_stable
# 提示：stable_check(BIG, "分组", "quicksort")
quick_stable = ____
heap_stable = ____
merge_stable = ____
stable_stable = ____
# ---- 验收 ----
assert quick_stable is False, "quicksort 是默认值，但不稳定 —— 这是本章第一大坑"
assert heap_stable is False, "heapsort 同样不稳定"
assert merge_stable is True, "mergesort 稳定"
assert stable_stable is True, "kind='stable' 稳定"
assert BIG.shape == (20_000, 2) and BIG["分组"].nunique() == 5
print("TODO(2) 通过 -> quicksort", quick_stable, "| heapsort", heap_stable,
      "| mergesort", merge_stable, "| stable", stable_stable)

In [ ]:
# TODO(3)：多列排序与混合升降序：
#           multi = 先按 设备类型 升序、再按 负荷值 降序
#           champ = multi 里每个 设备类型 的第一行（组内冠军）
#           champ_vals = champ 的 负荷值 列表
#           bad_asc = ascending 长度不匹配时的异常类型名
# 提示：dev.sort_values([CAT, VAL], ascending=[True, False])
#        multi.groupby(CAT, sort=False).head(1) ；probe(dev.sort_values, [CAT, VAL], ascending=[True])[1]
multi = ____
champ = ____
champ_vals = ____
bad_asc = ____
# ---- 验收 ----
assert multi.shape[0] == len(dev), "排序不改变行数"
assert champ.shape[0] == 5, f"5 个设备类型应各出 1 行，实际 {champ.shape[0]}"
assert champ_vals == [9999.0, 9999.0, 9999.0, 118.01, 103.81], f"实际 {champ_vals}"
assert bad_asc == "ValueError", "ascending 长度必须等于 by 长度"
assert bool(multi.groupby(CAT, sort=False)[VAL].first().tolist() == champ_vals), \
    "champ 的值应等于各组排序后的首值"
print("TODO(3) 通过 -> 冠军值", champ_vals, "| 长度不匹配 ->", bad_asc)

In [ ]:
# TODO(4)：排序后取行：loc 与 iloc 的分叉
#           top_iloc = by_val_desc.iloc[0][VAL]
#           top_loc  = by_val_desc.loc[0][VAL]
#           desc_ri  = by_val_desc.reset_index(drop=True)，其 loc[0][VAL] 存入 top_ri
# 提示：用 TODO(1) 排好的 by_val_desc
top_iloc = ____
top_loc = ____
desc_ri = ____
top_ri = ____
# ---- 验收 ----
assert top_iloc == 9999.0, "iloc[0] = 排完序的第一行 = 全局最大"
assert top_loc == 89.96, f"loc[0] 取的是索引标签 0，与排序无关，实际 {top_loc}"
assert top_ri == 9999.0, "reset_index(drop=True) 之后标签才是 0..n-1"
assert by_val_desc.loc[0, "记录ID"] == 31, "标签 0 那一行的记录 ID"
assert by_val_desc.iloc[0]["记录ID"] == 16, "排完序第一行的记录 ID"
assert probe(lambda: by_val_desc.iloc[0, VAL])[1] == "ValueError", \
    "iloc 的列也必须是整数位置，写列名直接 ValueError"
print("TODO(4) 通过 -> iloc", top_iloc, "| loc", top_loc, "| reset 后", top_ri)

In [ ]:
# TODO(5)：rank 的五种 method：
#           rank_tbl = {"average": [...], "min": [...], "max": [...], "first": [...], "dense": [...]}
#           dense_max = dense 法的最大值；min_max = min 法的最大值
# 提示：vals.rank(method="dense") ；用 METHODS 里的五个名字做循环键
rank_tbl = ____
dense_max = ____
min_max = ____
# ---- 验收 ----
assert rank_tbl["average"] == [1.0, 2.5, 2.5, 4.0, 6.0, 6.0, 6.0, 8.0], "默认 method"
assert rank_tbl["min"] == [1.0, 2.0, 2.0, 4.0, 5.0, 5.0, 5.0, 8.0]
assert rank_tbl["max"] == [1.0, 3.0, 3.0, 4.0, 7.0, 7.0, 7.0, 8.0]
assert rank_tbl["first"] == [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
assert rank_tbl["dense"] == [1.0, 2.0, 2.0, 3.0, 4.0, 4.0, 4.0, 5.0]
assert dense_max == 5.0, "dense 数的是「不同取值个数」"
assert min_max == 8.0, "min 数的是「行号」"
assert pd.Series([1, 2, 2]).rank().tolist() == [1.0, 2.5, 2.5], "默认 method 是 average"
assert int((vals.rank(method="average") % 1 != 0).sum()) == 2, "只有 average 会产出半个名次（两个并列的 20）"
print("TODO(5) 通过 -> dense max", dense_max, "| min max", min_max)

In [ ]:
# TODO(6)：组内名次：
#           dev["组内排名"] = 按 KEY 分组、对 VAL 降序的 min 法名次
#           dev["组内百分位"] = 同样分组、pct=True 的名次
#           rank_nan / n_first / n_top3 分别统计名次缺失数、第 1 名行数、前 3 名行数
# 提示：dev.groupby(KEY)[VAL].rank(ascending=False, method="min")
dev["组内排名"] = ____
dev["组内百分位"] = ____
rank_nan = ____
n_first = ____
n_top3 = ____
# ---- 验收 ----
assert rank_nan == 21, f"负荷值缺失 21 行，名次也应缺 21，实际 {rank_nan}"
assert n_first == 19, f"18 个台区 + 1 个并列冠军 = 19，实际 {n_first}"
assert n_top3 == 54, f"18 台区 x 3 = 54，实际 {n_top3}"
assert int(dev[KEY].nunique()) == 18
assert round(float(dev["组内百分位"].min()), 6) == 0.045455, "1/22 —— 最大的组有 22 行"
assert round(float(dev["组内百分位"].max()), 6) == 1.0, "每组都有一行拿到 1.0"
print("TODO(6) 通过 -> 缺失", rank_nan, "| 冠军行", n_first, "| 前 3 行", n_top3)

In [ ]:
# TODO(7)：nlargest / nsmallest 与 keep：
#           top5 = dev.nlargest(5, VAL) 的负荷值列表
#           bot5 = dev.nsmallest(5, VAL) 的负荷值列表
#           n_keep_all = dev.nlargest(3, VAL, keep="all") 的行数
# 提示：dev.nlargest(5, VAL)[VAL].tolist()
top5 = ____
bot5 = ____
n_keep_all = ____
# ---- 验收 ----
assert top5 == [9999.0, 9999.0, 9999.0, 9999.0, 5000.0], f"实际 {top5}"
assert bot5 == [-1.0, -1.0, -1.0, -1.0, 5.0], f"实际 {bot5}"
assert n_keep_all == 4, f"4 个哨兵 9999 并列第 1，keep='all' 全给，实际 {n_keep_all}"
assert len(dev.nlargest(3, VAL)) == 3, "keep='first'（默认）严格给 3 行"
assert int(dev.nlargest(3, VAL)[VAL].isna().sum()) == 0, "nlargest 永远不返回 NaN"
assert int(dev.nsmallest(3, VAL)[VAL].isna().sum()) == 0
print("TODO(7) 通过 -> top5", top5, "| n_keep_all", n_keep_all)

In [ ]:
# TODO(8)：可复现抽样：
#           s42a / s42b = 两次 dev.sample(5, random_state=42) 的索引列表
#           n_frac_small = dev.sample(frac=0.001, random_state=42) 的行数
#           n_frac_big   = dev.sample(frac=1.0, random_state=1) 的行数
#           bad_both = 同时给 n 和 frac 时的异常类型名
# 提示：frac 的行数是 round(n * frac)；probe(dev.sample, n=5, frac=0.1)
s42a = ____
s42b = ____
n_frac_small = ____
n_frac_big = ____
bad_both = ____
# ---- 验收 ----
assert s42a == s42b == [30, 116, 79, 127, 196], f"同一 random_state 必须完全复现，实际 {s42a}"
assert n_frac_small == 0, "270 x 0.001 = 0.27 → round 到 0，静默返回空表"
assert n_frac_big == 270, "frac=1.0 不放回 = 全表的一个随机排列"
assert bad_both == "ValueError", "n 与 frac 只能给一个"
assert probe(dev.sample, frac=2.0, random_state=42)[1] == "ValueError", "frac>1 必须 replace=True"
assert len(dev.sample(frac=2.0, replace=True, random_state=42)) == 540
print("TODO(8) 通过 ->", s42a, "| frac=0.001 行数", n_frac_small)

In [ ]:
# TODO(9)：sample(weights=...) 的两道门槛：
#           probe_neg    = 直接拿 VAL 当权重的不放回抽样的异常类型名
#           dpos         = dev 加上一列「权重」= VAL 截断负值并填 0
#           probe_norep  = 对 dpos 做 sample(10, weights="权重")（不放回）的异常类型名
#           ok_idx       = dpos.sample(10, weights="权重", replace=True, random_state=42) 的索引
#           ok_mean      = 上面这次加权抽样的 负荷值 均值（保留 4 位）
# 提示：dpos = dev.assign(权重=dev[VAL].clip(lower=0).fillna(0))
probe_neg = ____
dpos = ____
probe_norep = ____
sampled = ____
ok_idx = ____
ok_mean = ____
# ---- 验收 ----
assert probe_neg == "ValueError", "权重不能为负 —— 本表有 4 个 -1 哨兵"
assert probe_norep == "ValueError", "不放回加权采样要求 n * max(归一化权重) <= 1"
assert ok_idx == [56, 237, 154, 89, 46, 46, 40, 219, 89, 142], f"实际 {ok_idx}"
assert len(set(ok_idx)) < len(ok_idx), "replace=True 时同一行可能被抽中多次"
assert abs(float(dpos["权重"].sum()) - 94545.46) < 0.01, f"权重和 {float(dpos['权重'].sum())}"
assert round(float(dpos["权重"].max()), 2) == 9999.0
assert 10 * float(dpos["权重"].max()) / float(dpos["权重"].sum()) > 1, "这正是报错的原因"
assert ok_mean == 5666.517, f"加权抽样会把样本均值显著拉高，实际 {ok_mean}"
assert ok_mean > round(float(dev[VAL].mean()), 4), "远高于全表均值 379.6846"
print("TODO(9) 通过 ->", probe_neg, probe_norep, "| 加权均值", ok_mean)

In [ ]:
# TODO(10)：value_counts 的默认行为：
#           vc_cat = dev[CAT].value_counts()；vc_def = dev["缺陷类型"].value_counts()
#           n_def  = vc_def 的取值个数
#           vc2_shape = dev[[CAT, "缺陷等级"]].value_counts() 的形状
#           extra_na = 备注 列 dropna=False 与默认的条目数之差
# 提示：dev[CAT].value_counts() ；dev["备注"].value_counts(dropna=False).shape[0]
vc_cat = ____
vc_def = ____
n_def = ____
vc2_shape = ____
extra_na = ____
# ---- 验收 ----
assert vc_cat.shape[0] == 5, f"设备类型 5 类，实际 {vc_cat.shape[0]}"
assert vc_cat.to_dict() == {"避雷器": 59, "互感器": 58, "绝缘子": 57, "变压器": 55, "断路器": 41}, \
    f"实际 {vc_cat.to_dict()}"
assert vc_def.index[0] == "发热" and int(vc_def.iloc[0]) == 51, "默认按计数降序，最多的是发热 51"
assert int(vc_def.sum()) == 270, "本列无缺失，计数之和 = 行数"
assert n_def == 10, f"缺陷类型原样有 10 种写法（含 '锈蚀 ' 尾随空格），实际 {n_def}"
assert int(dev["缺陷类型"].str.strip().nunique()) == 9, "strip 后只剩 9 种"
assert vc2_shape == (15,), f"5 类型 x 3 等级共出现 15 个组合，实际 {vc2_shape}"
assert extra_na == 1, "备注有 196 个缺失，dropna=False 会多出 NaN 这一项"
assert round(float(vc_def.iloc[0]) / 270, 6) == round(float(vc_def.loc["发热"] / vc_def.sum()), 6)
print("TODO(10) 通过 -> 缺陷类型", n_def, "种 | 组合数", vc2_shape, "| 缺失项", extra_na)

---

## 进阶题

In [ ]:
# TODO(11)：综合①：一条链做出「每个台区的负荷冠军」并三方对账
#           champ_tbl = dev 按 VAL 降序（kind="stable"）排序后，groupby(KEY, sort=False).head(1)，
#                       取 [KEY, VAL] 两列，再 reset_index(drop=True)
#           champ_sum = champ_tbl 的 负荷值 之和（保留 2 位）
#           topn2     = topn_by_sort(dev, VAL, 2) 的行数
#           three_way = (champ_tbl 行数, dev[KEY].nunique(), 组内排名 == 1 的行数)
# 提示：dev.sort_values(VAL, ascending=False, kind="stable").groupby(KEY, sort=False).head(1)
champ_tbl = ____
champ_sum = ____
topn2 = ____
three_way = ____
# ---- 验收 ----
assert champ_tbl.shape == (18, 2), f"冠军表应为 (18, 2)，实际 {champ_tbl.shape}"
assert list(champ_tbl.columns) == [KEY, VAL], f"列名应为 [{KEY}, {VAL}]，实际 {list(champ_tbl.columns)}"
assert champ_tbl.index.tolist() == list(range(18)), "reset_index(drop=True) 后索引应为 0..17"
assert int(champ_tbl[VAL].isna().sum()) == 0, "每个台区至少有一条非缺失的负荷记录"
assert champ_sum == 46672.25, f"冠军负荷之和应为 46672.25，实际 {champ_sum}"
assert topn2 == 36, f"18 台区 x 2 = 36，实际 {topn2}"
assert three_way == (18, 18, 19), f"(冠军行 18, 台区 18, 第 1 名 19) 实际 {three_way}"
assert three_way[2] - three_way[0] == 1, "差的这 1 行 = STATION_E_03 的并列冠军（两个 9999）"
assert int(dev["组内排名"].isna().sum()) == 21, "名次缺失的行不会进入冠军表"
print("TODO(综合①) 通过 -> 冠军表", champ_tbl.shape, "| 之和", champ_sum, "| 三方", three_way)

In [ ]:
# TODO(12)：综合②：可复现采样报告
#           smp   = dev.sample(50, random_state=7)
#           strat = dev.groupby(CAT, group_keys=False).sample(2, random_state=7)
#           boot  = dev.sample(frac=0.5, replace=True, random_state=7)
#           report 字典键（顺序照抄）：
#             "样本行数" / "采样负荷均值"（4 位） / "全表负荷均值"（4 位）
#             / "分层行数" / "分层后组数" / "重采样行数" / "重采样重复索引数"
# 提示：每一项都要写死 random_state，报告才能复现；bootstrap 的重复索引数用 index.duplicated().sum()
smp = ____
strat = ____
boot = ____
report = ____
# ---- 验收 ----
assert report["样本行数"] == 50
assert report["采样负荷均值"] == 569.6152, f"实际 {report['采样负荷均值']}"
assert report["全表负荷均值"] == 379.6846, f"实际 {report['全表负荷均值']}"
assert report["分层行数"] == 10, "5 个设备类型 x 2 = 10"
assert report["分层后组数"] == 5
assert report["重采样行数"] == 135, f"270 x 0.5 = 135，实际 {report['重采样行数']}"
assert report["重采样重复索引数"] == 28, f"实际 {report['重采样重复索引数']}"
assert report["采样负荷均值"] > report["全表负荷均值"], \
    "50 行的样本均值偏离全量均值很远 —— 这正是「抽样要看的东西」"
assert len(report) == 7 and list(report)[0] == "样本行数"
print("TODO(综合②) 通过：")
for k, v in report.items():
    print(f"  {k:14s} = {v}")

---

## 自查清单

跑完全部 `assert` 之后，回看这几件事：

1. 每个 `random_state` 都写死了吗？（不写就不可复现）
2. 涉及并列的排序都加 `kind="stable"` 了吗？
3. `rank` 的 `method` 是按业务语义选的吗？（名次用 `min`，等级用 `dense`）
4. `value_counts` 需要看缺失时，加 `dropna=False` 了吗？
5. 交付前索引都复位了吗？（`reset_index(drop=True)` 或 `ignore_index=True`）